# Workable Code

In [203]:
%load_ext autoreload
%autoreload 2 

import pandas as pd
from src.utils import clean_customer_gdf_coordinates
from src.H3SpatialClusterer import H3SpatialClusterer  
import json
from src.get_data import DataFetcher, get_processed_data, get_geojson_data
import pickle
from src.utils import filter_cluster_result_dict
from src.plot_utils import plot_geojson_territory_heatmap

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Test Pipeline

In [ ]:
## Fetch Data from DB
# fetcher = DataFetcher(logger=my_logger, input_dir="./input", sql_dir="_sql")
# results = fetcher.fetch_all()

In [204]:
OUTPUT_JSON_MAP = "./output/h3_clusters_map.geojson"
OUTPUT_JSON_MAP_FILTER = "./output/h3_clusters_map_filter.geojson"
OUTPUT_RESULTS_PICKLE = "./output/cluster_results.pickle"
OUTPUT_MAP_FILTERED = './output/map/test_nigeria_clusters_filter.html' 

# CUSTOMER_DENSITY_THRESHOLD_HIGH = 200
# CUSTOMER_DENSITY_THRESHOLD_LOW = 10
# MAX_H3_RESOLUTION_URBAN = 12
# MERGE_SEARCH_RADIUS = 1
# POP_DENSITY_HIGH = 5000
# POP_DENSITY_MEDIUM = 1000


pilot_sps_lists = ['1647024','1647113','1647122']
stock_point_id = str(pilot_sps_lists[1])

In [205]:
#### Get Data
lgas_gdf,  sp_dim_df,  stock_point_lga_map, customers_gdf = get_processed_data()

⚠️ Warning: Dropped 806 customer records with coordinates outside Nigeria.
✅ Customer data cleaning complete. 28209 of 29015 records retained.


In [ ]:
5# sp_dim_df.query("stock_point_name in ['OmniHub Oyigbo Rivers - LAMDA GLOBAL','OmniHub Apapa Lagos - CAUSEWAY','OmniHub Egbeda Oyo - Vizazi']")#['stock_point_name'].values[0]

In [ ]:
lgas_gdf.query('state_name == "Oyo"').area_km2.sum()

np.float64(2874.4894999999997)

In [ ]:
# 1. Initialize with real dataframes
clusterer = H3SpatialClusterer(
    lga_gdf=lgas_gdf,
    sp_dim_df=sp_dim_df, 
    stock_point_lga_map=stock_point_lga_map,
    customers_gdf=customers_gdf
)

In [ ]:
lgas_gdf.head(1)

In [ ]:
# # 3. Run complete pipeline
'''
This will return a dictionary with the following keys: ['territories', 'grid_results', 'assignments', 'optimized_clusters', 'statistics', 'territory_version']
1. territories: A dictionary of stock point territories 
        # Dict[stock_point_id, {
            'polygon': Union[Polygon, MultiPolygon],
            'lga_ids': List[str],
            'is_contiguous': bool,
            'sub_territories': List[Polygon],
            'total_area_km2': float,
            'territory_version': str
        }]

2. grid_results: A dictionary of grid results for each territory
        Dict[stock_point_id, {
                'h3_resolution': int,
                'h3_cells': Set[str],
                'clipped_cells': Set[str], 
                'cell_geometries': Dict[str, Polygon],
                'territory_coverage': float
        }]
        
3. assignments: A dictionary of customer assignments to stock points
        Dict[stock_point_id, assignments_gdf with columns:
                ['customer_id', 
                'cluster_id', 
                'h3_cell_id', 
                'assignment_confidence', 
                'assignment_tier',
                'geometry']]
        
4. optimized_clusters: A dictionary of optimized clusters for each territory

5. statistics: A dictionary of statistics for each territory

6. territory_version: The version of the territory used in the clustering
'''

# # RESULTS: ETA 2MINS
# results = clusterer.process_all_stock_points(territory_version="v1.2")

# # Save Results as pickle file
# with open(OUTPUT_RESULTS_PICKLE, 'wb') as f:
#     pickle.dump(results, f)


In [ ]:
# Read results from the pickle file
with open(OUTPUT_RESULTS_PICKLE, 'rb') as f:
    results = pickle.load(f)

filtered_result = filter_cluster_result_dict(results, pilot_sps_lists)

In [ ]:
assignments = results['assignments'][stock_point_id]
assignments.head(1)

In [ ]:
# stock_point_id = str(pilot_sps_lists[0])
# results['optimized_clusters'][stock_point_id].keys()
# ['territories', 'grid_results', 'assignments', 'optimized_clusters', 'statistics', 'territory_version'] 

territories = results['territories'][stock_point_id]
# ['polygon', 'lga_ids', 'is_contiguous', 'sub_territories', 
# 'total_area_km2', 'territory_version', 'lga_count', 'validation_status']

grid_data = results['grid_results'][stock_point_id]
# ['h3_resolution', 'h3_cells', 'clipped_cells', 'cell_geometries', 'territory_coverage']

assignments = results['assignments'][stock_point_id]
# ['customer_id', 'cluster_id', 'h3_cell_id', 'assignment_confidence',  'assignment_tier', 'geometry']

optimized_clusters = results['optimized_clusters'][stock_point_id]
# statistics = results['statistics'][stock_point_id]
# territory_version = results['territory_version']
# results['territories'][stock_point_id]
# results['assignments'][stock_point_id].sample(2)
# results['optimized_clusters'][stock_point_id]#.sample(2)

In [ ]:
# assignment cell summary: Dataframe: cluster_id, n_cusomter
# TO-DO ADD SUMMARY BY 
try:
    assignment_cell_summary =  assignments.groupby('cluster_id')['customer_id'].count().reset_index(name='n_customers')
except:
    assignment_cell_summary = pd.DataFrame(columns = ['cluster_id','n_customers'])

In [ ]:
results.get('grid_results', {}).get(stock_point_id, {}).keys()

In [ ]:

# 4. Export for deployment: ETA: 1 Min
csv_files = clusterer.export_results(results, output_format="csv")

# Stock Point Assignment Summary
sp_assignment_summary = (csv_files['assignments']
                            .groupby(['stock_point_id','cluster_id'])['customer_id'].count()
                            .reset_index(name='n_customers')
                            .rename({'cluster_id':'cell'}, axis=1)
                        )

# Stock Point Coverage - Assignment Summary
sp_coverage_assignment_summary = (csv_files['territory_cells']
                                .merge(sp_assignment_summary, how='left', on=['stock_point_id','cell'])
                                .fillna({'n_customers':0})
                                ) 




# csv_files['clusters'].to_csv('clusters_output.csv', index=False)
# csv_files['assignments'].to_csv('customer_assignments.csv', index=False)




In [220]:
csv_files['clusters'].query('stock_point_id == @stock_point_id').head(1)

,cluster_id,h3_resolution,h3_cells,customer_count,parent_cluster_id,stock_point_id
977,8858826c91fffff,8,[8858826c91fffff],35,None,1647113


In [222]:
sp_coverage_assignment_summary.query('stock_point_id == @stock_point_id').head(1)

,cell,stock_point_id,h3_resolution,n_customers
41137,88589cd64bfffff,1647113,8,1.0


In [223]:
sp_assignment_summary.query('stock_point_id == @stock_point_id').head(1)

,stock_point_id,cell,n_customers
977,1647113,8858826c91fffff,35


In [ ]:
# 5. Export for GeoJSON: ETA: 1 Min

# sql_statements and geojson_map
geojson_map = clusterer.export_results(results, output_format="geojson")
geojson_map_filtered = clusterer.export_results(filtered_result, output_format="geojson") 

# Save full geojson_map data
try:
    with open(OUTPUT_JSON_MAP, 'w') as f:
        f.write(geojson_map)
    print(f"GeoJSON map successfully saved to {OUTPUT_JSON_MAP}")
except IOError as e:
    print(f"Error saving GeoJSON map to file: {e}")
    

# Save filtered geojson_map data    
try:
    with open(OUTPUT_JSON_MAP_FILTER, 'w') as f:
        f.write(geojson_map_filtered)
    print(f"GeoJSON map successfully saved to {OUTPUT_JSON_MAP_FILTER}")
except IOError as e:
    print(f"Error saving GeoJSON map to file: {e}")
    


In [ ]:
## PLOT 
 
_ = plot_geojson_territory_heatmap(geojson_path = OUTPUT_JSON_MAP_FILTER, output_path = output_path)

Successfully loaded GeoJSON data from ./output/h3_clusters_map_filter.geojson


In [ ]:


# 5. Get processing summary
summary = clusterer.get_processing_summary()

# # 6. Deploy to database
# for sql in sql_statements:
#     database.execute(sql)

# # 7. Save visualization files
# with open('nigeria_clusters.geojson', 'w') as f:
#     f.write(geojson_map)
    
# csv_files['clusters'].to_csv('clusters_output.csv', index=False)
# csv_files['assignments'].to_csv('customer_assignments.csv', index=False)

In [ ]:
summary

### Diagonistics

In [ ]:
import geopandas as gpd
import pandas as pd

def reassign_customers_with_optimized_clusters(
    assignments: Dict[str, gpd.GeoDataFrame],
    optimized_clusters: Dict[str, pd.DataFrame]
) -> Dict[str, gpd.GeoDataFrame]:
    """
    Reassigns customers to optimized cluster IDs based on the output of optimize_clusters.
    
    Args:
        assignments: Dict mapping stock_point_id to GeoDataFrame with customer data
                    (columns include 'cluster_id', 'geometry').
        optimized_clusters: Dict mapping stock_point_id to DataFrame with optimized clusters
                           (columns: 'cluster_id', 'h3_resolution', 'h3_cells', 'customer_count', 'parent_cluster_id').
    
    Returns:
        Dict[stock_point_id, GeoDataFrame with updated 'cluster_id' for each customer].
    """
    print("🔄 Reassigning customers to optimized cluster IDs...")
    
    reassigned_assignments = {}
    
    for stock_point_id, assignments_gdf in assignments.items():
        print(f"Processing stock point {stock_point_id}...")
        
        if assignments_gdf.empty or stock_point_id not in optimized_clusters:
            reassigned_assignments[stock_point_id] = assignments_gdf.copy()
            continue
        
        optimized_df = optimized_clusters[stock_point_id]
        if optimized_df.empty:
            reassigned_assignments[stock_point_id] = assignments_gdf.copy()
            continue
        
        # Create a mapping from original H3 cell to optimized cluster_id
        cell_to_cluster_id = {}
        for _, cluster in optimized_df.iterrows():
            cluster_id = cluster['cluster_id']
            for cell in cluster['h3_cells']:
                cell_to_cluster_id[cell] = cluster_id
        
        # Copy the input GeoDataFrame to preserve all columns
        reassigned_gdf = assignments_gdf.copy()
        
        # Reassign cluster_id for each customer
        reassigned_gdf['optimized_cluster_id'] = None
        for idx, customer in reassigned_gdf.iterrows():
            original_cell = customer['cluster_id']
            if original_cell in cell_to_cluster_id:
                reassigned_gdf.at[idx, 'optimized_cluster_id'] = cell_to_cluster_id[original_cell]
            else:
                # If the original cell isn't in optimized clusters, try reassigning based on location
                lat, lng = customer.geometry.y, customer.geometry.x
                customer_resolution = optimized_df['h3_resolution'].max()  # Use highest resolution
                customer_cell = h3.latlng_to_cell(lat, lng, customer_resolution)
                # Find the optimized cluster containing this cell
                for _, cluster in optimized_df.iterrows():
                    if customer_cell in cluster['h3_cells'] or h3.cell_to_parent(customer_cell, cluster['h3_resolution']) in cluster['h3_cells']:
                        reassigned_gdf.at[idx, 'optimized_cluster_id'] = cluster['cluster_id']
                        break
                if reassigned_gdf.at[idx, 'optimized_cluster_id'] is None:
                    print(f"⚠️ Customer at index {idx} could not be reassigned, keeping original cluster_id {original_cell}")
                    reassigned_gdf.at[idx, 'optimized_cluster_id'] = original_cell
        
        reassigned_assignments[stock_point_id] = reassigned_gdf
        print(f"✅ Reassigned {len(reassigned_gdf)} customers for stock point {stock_point_id}")
    
    print("🏁 Customer reassignment complete")
    return reassigned_assignments

In [ ]:
# Get LGA IDs for this stock point
stock_point_id = 1647108
lga_ids = clusterer.stock_point_lga_map[
    clusterer.stock_point_lga_map['stock_point_id'] == stock_point_id
]['lga_id'].tolist()

In [ ]:
# lga_ids
if not lga_ids:
    print(f"⚠️ Warning: No LGAs found for stock_point_id {stock_point_id}")
    # continue

# Get LGA geometries
territory_lgas = clusterer.lgas[clusterer.lgas['lga_id'].isin(lga_ids)].copy()
territory_lgas

In [ ]:
lga_gdf.shape # (115, 18)

lga_gdf_2 = lga_gdf.dropna(subset=['geometry'])

lga_gdf_2[lga_gdf_2['lga_id'].isin(lga_ids)].copy()

# Pipeline 0

#### 00. Utils

In [ ]:
import logging
import sys
from contextlib import contextmanager
import os

@contextmanager
def suppress_stdout():
    """Context manager to temporarily suppress print output."""
    with open(os.devnull, 'w') as devnull:
        old_stdout = sys.stdout
        sys.stdout = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import numpy as np # For creating example NaN data
from typing import Union, List

def validate_coordinate_within_nigeria(lat: float, lng: float) -> bool:
    """
    Validates if a given latitude and longitude fall within Nigeria's
    approximate geographical bounds.

    Args:
        lat (float): The latitude of the point.
        lng (float): The longitude of the point.

    Returns:
        bool: True if coordinates are within Nigeria bounds and not NaN, False otherwise.
    """
    nigeria_bounds = {
        'min_lat': 4.0,   'max_lat': 14.0,
        'min_lng': 2.5,   'max_lng': 15.0
    }
    
    # Ensure lat/lng are not NaN before comparison
    if pd.isna(lat) or pd.isna(lng):
        return False

    return (nigeria_bounds['min_lat'] <= lat <= nigeria_bounds['max_lat'] and
            nigeria_bounds['min_lng'] <= lng <= nigeria_bounds['max_lng'])

def clean_customer_gdf_coordinates(customers_gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Cleans a GeoDataFrame of customer data by filtering out records with
    invalid geometries or coordinates outside Nigeria.

    Args:
        customers_gdf (gpd.GeoDataFrame): The input GeoDataFrame with customer data,
                                         expected to have a 'geometry' column of Point types.

    Returns:
        gpd.GeoDataFrame: A new GeoDataFrame with cleaned customer data.
    """
    original_count = len(customers_gdf)
    
    # Step 1: Drop rows where 'geometry' itself is NaN/None
    customers_cleaned_geom = customers_gdf.dropna(subset=['geometry']).copy()
    dropped_geom_count = original_count - len(customers_cleaned_geom)
    if dropped_geom_count > 0:
        print(f"⚠️ Warning: Dropped {dropped_geom_count} customer records due to missing geometry.")

    # Step 2: Validate coordinates using the standalone helper function
    # We apply the validation function to each geometry's coordinates
    valid_coords_mask = customers_cleaned_geom.geometry.apply(
        lambda p: validate_coordinate_within_nigeria(p.y, p.x) if p is not None else False
    )
    
    customers_final = customers_cleaned_geom[valid_coords_mask].copy()
    dropped_oob_count = len(customers_cleaned_geom) - len(customers_final)
    if dropped_oob_count > 0:
        print(f"⚠️ Warning: Dropped {dropped_oob_count} customer records with coordinates outside Nigeria.")
        
    print(f"✅ Customer data cleaning complete. {len(customers_final)} of {original_count} records retained.")
    return customers_final


In [ ]:
import folium
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon # Ensure MultiPolygon is imported
import h3 # This needs to be the h3 module that is loaded
from typing import List, Set # Already there, just for completeness

def plot_territory(territory_polygons: List[Polygon], stock_point_id: int):
    """
    Plots the territory polygons for a given stock point.
    
    Args:
        territory_polygons (List[Polygon]): List of Shapely Polygon objects defining the territory.
        resolution (int): H3 resolution used for the territory.
    """
    import matplotlib.pyplot as plt
    # Assuming territory_polygons is your List[Polygon]
    territory_gdf_to_plot = gpd.GeoDataFrame(geometry=territory_polygons, crs="EPSG:4326")
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    territory_gdf_to_plot.plot(ax=ax, color='blue', edgecolor='black', alpha=0.5)
    ax.set_title(f"Territory for Stock Point {stock_point_id}")
    plt.axis('off') # Optional: removes axis ticks and labels for cleaner map
    plt.show()
    
# plot_territory(territory_polygons, stock_point_id=1647113)

In [ ]:


# Ensure you have these imports at the top of your script if not already:
# import folium
# import geopandas as gpd
# from shapely.geometry import Polygon, MultiPolygon
# import h3


def plot_territory_and_h3_overlay(
    territory_polygons: List[Polygon],
    territory_cells: Set[str],
    title: str = "Territory with H3 Grid Overlay",
    h3_resolution: int = None
) -> folium.Map:
    """
    Plots territory polygons and overlays H3 cells on an interactive Folium map.
    This version manually constructs H3 cell polygons due to a specific H3-py module issue.

    Args:
        territory_polygons (List[Polygon]): A list of Shapely Polygon objects defining the territory.
        territory_cells (Set[str]): A set of H3 cell IDs covering the territory.
        title (str): The title for the map.
        h3_resolution (int, optional): The H3 resolution used, for display in title/legend.

    Returns:
        folium.Map: An interactive Folium map object.
    """
    if not territory_polygons:
        print("No territory polygons provided for plotting.")
        return folium.Map(location=[6.5244, 3.3792], zoom_start=9) # Default to Lagos center

    # 1. Convert territory polygons to GeoDataFrame for easier plotting
    territory_gdf = gpd.GeoDataFrame(geometry=territory_polygons, crs="EPSG:4326")

    # 2. Determine map center (approximate centroid of the territory)
    united_territory = territory_gdf.union_all()
    # united_territory = territory_gdf.unary_union
    
    if united_territory.is_empty or not united_territory.is_valid:
        map_center = [6.5244, 3.3792]
        zoom_start = 9
    else:
        map_center = [united_territory.centroid.y, united_territory.centroid.x]
        min_lon, min_lat, max_lon, max_lat = united_territory.bounds
        if (max_lon - min_lon) < 0.1 and (max_lat - min_lat) < 0.1:
            zoom_start = 13
        elif (max_lon - min_lon) < 0.5 and (max_lat - min_lat) < 0.5:
            zoom_start = 12
        else:
            zoom_start = 10

    # 3. Initialize Folium Map
    m = folium.Map(location=map_center, zoom_start=zoom_start, control_scale=True)

    # Add a title
    title_html = f'<h3 align="center" style="font-size:20px"><b>{title}</b></h3>'
    if h3_resolution:
        title_html = f'<h3 align="center" style="font-size:20px"><b>{title} (H3 Res: {h3_resolution})</b></h3>'
    m.get_root().html.add_child(folium.Element(title_html))

    # 4. Add Territory Polygons to the map
    folium.GeoJson(
        territory_gdf.to_json(),
        name="Territory Boundaries",
        style_function=lambda x: {
            "fillColor": "#1a75ff",
            "color": "black",
            "weight": 2,
            "fillOpacity": 0.2,
        },
    ).add_to(m)

    # 5. Manually construct GeoJSON for H3 Grid Cells using available functions
    h3_polygons = []
    for cell in territory_cells:
        # h3.cell_to_boundary returns a list of (lat, lng) tuples
        boundary_lat_lng = h3.cell_to_boundary(cell)
        # Shapely.Polygon expects coordinates in (lng, lat) order
        boundary_lng_lat = [(lng, lat) for lat, lng in boundary_lat_lng]
        
        # Create a Shapely Polygon from the boundary
        h3_polygons.append(Polygon(boundary_lng_lat))

    # Convert the list of Shapely Polygons into a GeoDataFrame, then to GeoJSON
    if h3_polygons: # Ensure there are polygons before creating GeoDataFrame
        h3_cells_gdf = gpd.GeoDataFrame(geometry=h3_polygons, crs="EPSG:4326")
        h3_geojson_data = h3_cells_gdf.to_json()
    else:
        h3_geojson_data = {"type": "FeatureCollection", "features": []} # Empty GeoJSON

    folium.GeoJson(
        h3_geojson_data,
        name="H3 Grid Cells",
        style_function=lambda x: {
            "fillColor": "red",
            "color": "red",
            "weight": 1,
            "fillOpacity": 0.1,
        },
    ).add_to(m)
    
    # 6. Add Layer Control
    folium.LayerControl().add_to(m)

    return m

#### 01. Data

In [ ]:
import pandas as pd
import geopandas as gpd
import pickle
# loading preprocessed data
lga_gdf = pickle.load(open('./input/lga_gdf.pickle', 'rb'))
sp_dim_df = pd.read_feather('./input/df_sp_dim.feather') 
sp_lga_df = pd.read_feather('./input/df_sp_location_mapping.feather')
customers_df = pd.read_feather('./input/df_sp_customers.feather')

## Data Pre-Processing
lga_data = (lga_gdf.copy()
            .rename(columns={'lga_name': 'name'})
            .assign(population_density = 0)
            [['lga_id', 'name', 'population_density', 'geometry','area_km2']]
            .dropna()
            )
lgas_gdf = gpd.GeoDataFrame(lga_data, crs="EPSG:4326")

customer_data = customers_df.copy()[["customer_id", "stock_point_id", "longitude", "latitude"]]
customers_gdf = gpd.GeoDataFrame(customer_data, 
                                 geometry=gpd.points_from_xy(customer_data['longitude'], customer_data['latitude']), 
                                 crs="EPSG:4326")
# Clean the customer data using the new utility function
customers_gdf = clean_customer_gdf_coordinates(customers_gdf)

stock_point_lga_map = sp_lga_df.copy()[["stock_point_id", "lga_id"]]

CUSTOMER_DENSITY_THRESHOLD_HIGH = 5

In [ ]:
dict_dfs = {'lga_gdf':lga_gdf, 
            'sp_dim_df':sp_dim_df, 
            'stock_point_lga_map':stock_point_lga_map, 
            'customers_gdf':customers_gdf}
for df in dict_dfs.keys():
    # Print df name and columns names
    print(f"DataFrame: {df}")
    print("Columns:", dict_dfs[df].columns.tolist())

### Main Class

In [ ]:
import pandas as pd
import geopandas as gpd
import h3
from shapely.geometry import Polygon, MultiPolygon
from typing import List, Dict, Any, Set

# --- Configuration Constants based on the Specification ---
# Optimization thresholds
CUSTOMER_DENSITY_THRESHOLD_HIGH = 200 # Split cluster if customer count > this
CUSTOMER_DENSITY_THRESHOLD_LOW = 10   # Merge cluster if customer count < this
MAX_H3_RESOLUTION_URBAN = 12          # Max resolution for splitting
MERGE_SEARCH_RADIUS = 1               # H3 grid disk radius for finding merge candidates

# Adaptive Resolution thresholds
POP_DENSITY_HIGH = 5000 # For megacities like Lagos
POP_DENSITY_MEDIUM = 1000 # For other urban areas



class H3SpatialClusterer:
    """
    Implements a complete, boundary-constrained H3 spatial clustering workflow,
    including adaptive resolution, splitting, and merging as per the technical spec.
    """

    def __init__(
        self,
        lgas_gdf: gpd.GeoDataFrame,
        customers_gdf: gpd.GeoDataFrame,
        stock_point_lga_map: pd.DataFrame,
    ):
        """Initializes the clusterer with necessary geodata."""
        self.lgas = lgas_gdf
        self.customers = customers_gdf
        self.stock_point_lga_map = stock_point_lga_map
        print("✅ H3SpatialClusterer initialized.")

    ## NEW - FROM PHASE 2
    def _calculate_adaptive_resolution(self, lga_ids: List[str]) -> int:
        """Calculate adaptive H3 resolution based on population density"""
        if not lga_ids:
            return 8  # Default rural resolution
        
        # Get LGA data for density calculation
        relevant_lgas = self.lgas[self.lgas['lga_id'].isin(lga_ids)]
        
        if relevant_lgas.empty or 'population_density' not in relevant_lgas.columns:
            return 8
        
        # Calculate weighted average density
        if 'area_km2' in relevant_lgas.columns:
            total_area = relevant_lgas['area_km2'].sum()
            if total_area > 0:
                weighted_density = (relevant_lgas['population_density'] * relevant_lgas['area_km2']).sum() / total_area
            else:
                weighted_density = relevant_lgas['population_density'].mean()
        else:
            weighted_density = relevant_lgas['population_density'].mean()
        
        # Apply resolution logic from spec
        if weighted_density > POP_DENSITY_HIGH:  # 5000
            return 11  # Lagos megacity
        elif weighted_density > POP_DENSITY_MEDIUM:  # 1000  
            return 10  # Medium density
        else:
            return 8   # Rural default
        
        return 8 
    
    def process_stock_point(self, stock_point_id: int, territory_version: str = "v1.2") -> Dict[str, Any]:
        """Executes the full clustering pipeline for a single stock point."""
        print(f"\n--- Processing Stock Point ID: {stock_point_id} ---")
        
        territory_polygons = self._get_territory_polygons(stock_point_id)
        if not territory_polygons: return {"clusters": pd.DataFrame(), "assignments": pd.DataFrame()}
        resolution = self._get_adaptive_resolution(stock_point_id)
        territory_cells = self._generate_h3_grid(territory_polygons, resolution)
        print(f"🗺️  Phase 1/2: Territory defined at H3 res {resolution} with {len(territory_cells)} cells.")

        customers_sp = self.customers[self.customers["stock_point_id"] == stock_point_id].copy()
        if customers_sp.empty:
            print("No customers for this stock point. Skipping.")
            return {"clusters": pd.DataFrame(), "assignments": pd.DataFrame()}

        customers_sp['base_cell'] = customers_sp.geometry.apply(lambda p: h3.latlng_to_cell(p.y, p.x, resolution))
        initial_counts = customers_sp['base_cell'].value_counts().to_frame(name='customer_count')
        print(f"🧑‍🤝‍🧑 Phase 3: Initially assigned {len(customers_sp)} customers to {len(initial_counts)} cells.")

        opt_results = self._optimize_clusters(initial_counts, resolution, territory_cells)
        final_clusters = opt_results["clusters"]
        merge_map = opt_results["merge_map"]
        print(f"⚙️  Phase 4: Optimized into {len(final_clusters)} clusters. Merged {len(merge_map)} sparse cells.")

        def get_final_cluster_id(row):
            base_cell = row['base_cell']
            final_id = merge_map.get(base_cell, base_cell)
            cluster = final_clusters[final_clusters['cluster_id'] == final_id]
            if not cluster.empty and cluster.iloc[0]['parent_cluster_id'] is not None:
                child_res = cluster.iloc[0]['h3_resolution']
                return h3.latlng_to_cell(row.geometry.y, row.geometry.x, child_res)
            return final_id

        customers_sp['cluster_id'] = customers_sp.apply(get_final_cluster_id, axis=1)
        
        final_counts = customers_sp['cluster_id'].value_counts().reset_index(name='customer_count')
        output_clusters = final_clusters.drop(columns=['customer_count'], errors='ignore').merge(final_counts, on='cluster_id', how='left').fillna(0)
        output_clusters['customer_count'] = output_clusters['customer_count'].astype(int)
        output_clusters['stock_point_id'] = stock_point_id
        output_clusters['territory_version'] = territory_version
        output_clusters = output_clusters.rename(columns={'cluster_id': 'id'})
        
        assignments = customers_sp[["customer_id", "cluster_id"]]
        print(f"--- ✅ Processing for Stock Point {stock_point_id} complete. ---")
        return {"clusters": output_clusters, "assignments": assignments}


### Dev - Case By Case

In [ ]:
# h3_spatial_clustering_v2.py

import pandas as pd
import geopandas as gpd
import h3
from shapely.geometry import Polygon, MultiPolygon
from typing import List, Dict, Any, Set

# --- Configuration Constants based on the Specification ---
# Optimization thresholds 
CUSTOMER_DENSITY_THRESHOLD_HIGH = 200 # Split cluster if customer count > this
CUSTOMER_DENSITY_THRESHOLD_LOW = 10   # Merge cluster if customer count < this
MAX_H3_RESOLUTION_URBAN = 12          # Max resolution for splitting
MERGE_SEARCH_RADIUS = 1               # H3 grid disk radius for finding merge candidates

# Adaptive Resolution thresholds
POP_DENSITY_HIGH = 5000 # For megacities like Lagos
POP_DENSITY_MEDIUM = 1000 # For other urban areas


stock_point_id = 1647113 # Example stock point ID
clusterer = H3SpatialClusterer(lgas_gdf, customers_gdf, stock_point_lga_map)

#### Phase 1  
Output:  
   - territories
   - cw_terriroty

In [ ]:
# Phase 1: Territory Definition and Validation
def define_territories(self) -> Dict[str, Dict[str, Any]]:
    """
    Phase 1: Territory Definition and Validation
    
    Creates boundary-constrained territories for each stock point using LGA geometries.
    Handles non-contiguous territories and validates geometric integrity.
    
    Returns:
        Dict[stock_point_id, {
            'polygon': Union[Polygon, MultiPolygon],
            'lga_ids': List[str],
            'is_contiguous': bool,
            'sub_territories': List[Polygon],
            'total_area_km2': float,
            'territory_version': str
        }]
    """
    print("🗺️ Phase 1: Starting territory definition...")
    
    territories = {}
    
    # Get all unique stock points
    stock_points = self.stock_point_lga_map['stock_point_id'].unique()
    
    for stock_point_id in stock_points:
        print(f"Processing territory for stock point {stock_point_id}...")
        
        # Get LGA IDs for this stock point
        lga_ids = self.stock_point_lga_map[
            self.stock_point_lga_map['stock_point_id'] == stock_point_id
        ]['lga_id'].tolist()
        
        if not lga_ids:
            print(f"⚠️ Warning: No LGAs found for stock_point_id {stock_point_id}")
            continue
        
        # Get LGA geometries
        territory_lgas = self.lgas[self.lgas['lga_id'].isin(lga_ids)].copy()
        
        if territory_lgas.empty:
            print(f"⚠️ Warning: No LGA geometries found for stock_point_id {stock_point_id}")
            continue
        
        # Validate individual LGA geometries
        validated_geometries = []
        for idx, lga in territory_lgas.iterrows():
            geom = lga['geometry']
            if not geom.is_valid:
                print(f"🔧 Fixing invalid geometry for LGA {lga['lga_id']}")
                geom = geom.buffer(0)  # Fix self-intersections
            validated_geometries.append(geom)
        
        # Create unified territory
        if len(validated_geometries) == 1:
            unified_territory = validated_geometries[0]
        else:
            # Union all LGA geometries
            from shapely.ops import unary_union
            unified_territory = unary_union(validated_geometries)
        
        # Final validation of unified territory
        if not unified_territory.is_valid:
            print(f"🔧 Fixing unified territory geometry for stock point {stock_point_id}")
            unified_territory = unified_territory.buffer(0)
        
        # Handle non-contiguous territories
        is_contiguous = isinstance(unified_territory, Polygon)
        sub_territories = []
        
        if isinstance(unified_territory, Polygon):
            sub_territories = [unified_territory]
        elif isinstance(unified_territory, MultiPolygon):
            sub_territories = list(unified_territory.geoms)
            print(f"📍 Non-contiguous territory detected: {len(sub_territories)} sub-territories")
        else:
            print(f"⚠️ Unexpected geometry type for stock point {stock_point_id}: {type(unified_territory)}")
            continue
        
        # Calculate total area
        total_area_km2 = sum(territory_lgas['area_km2']) if 'area_km2' in territory_lgas.columns else 0
        
        # Store territory information
        territories[str(stock_point_id)] = {
            'polygon': unified_territory,
            'lga_ids': lga_ids,
            'is_contiguous': is_contiguous,
            'sub_territories': sub_territories,
            'total_area_km2': total_area_km2,
            'territory_version': 'v1.2',
            'lga_count': len(lga_ids),
            'validation_status': 'valid'
        }
        
        # print(f"✅ Territory defined: {len(lga_ids)} LGAs, {len(sub_territories)} sub-territories, {total_area_km2:.1f} km²")
    
    print(f"🏁 Phase 1 complete: {len(territories)} territories defined")
    return territories

In [ ]:
# Phase 1
with suppress_stdout():
    territories = define_territories(self=clusterer)

# Non - Contiguous territories
for sp_id, territory in territories.items():
    if not territory['is_contiguous']:
        print(f"Stock Point {sp_id} has non-contiguous territory with {len(territory['sub_territories'])} sub-territories.")
        for idx, sub_territory in enumerate(territory['sub_territories']):
            print(f"  Sub-territory {idx+1}: Area = {sub_territory.area:.2f} sq. units")
            
# Checking CuaseWay
cw_terriroty = {'1647113':territories['1647113']}  # Display the keys of the phase_1 dictionary   
cw_terriroty         

#### Phase 2

In [ ]:
from shapely.errors import GEOSException
from shapely.validation import make_valid
from shapely.geometry import Polygon
from shapely.ops import unary_union
import h3
import math

def generate_h3_grids(self, territories: Dict[str, Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    """
    Phase 2: H3 Grid Generation with Adaptive Resolution
    
    Generates boundary-clipped H3 hexagons for each territory with density-based resolution.
    
    Args:
        territories: Dict mapping stock_point_id to territory data, including 'sub_territories' (list of shapely Polygons)
                     and 'polygon' (shapely Polygon for the entire territory).
    
    Returns:
        Dict[stock_point_id, {
            'h3_resolution': int,
            'h3_cells': Set[str],
            'clipped_cells': Set[str], 
            'cell_geometries': Dict[str, Polygon],
            'territory_coverage': float
        }]
    """
    print("🔶 Phase 2: Starting H3 grid generation...")
    
    grid_results = {}
    
    for stock_point_id, territory_data in territories.items():
        print(f"Generating H3 grid for stock point {stock_point_id}...")
        
        # Calculate adaptive resolution
        resolution = self._calculate_adaptive_resolution(territory_data['lga_ids'])
        
        # Initialize containers
        all_h3_cells = set()
        clipped_cells = set()
        cell_geometries = {}
        
        for sub_territory_idx, sub_territory in enumerate(territory_data['sub_territories']):
            try:
                # Convert sub-territory to H3 shape and generate cells
                h3_shape = h3.geo_to_h3shape(sub_territory.__geo_interface__)
                sub_cells = set(h3.polygon_to_cells(h3_shape, resolution))
                all_h3_cells.update(sub_cells)
                print(f"Sub-territory {sub_territory_idx} generated {len(sub_cells)} H3 cells")
                
                # Process each H3 cell
                for cell_id in sub_cells:
                    try:
                        # Validate cell ID
                        if not h3.is_valid_cell(cell_id):
                            print(f"⚠️ Invalid H3 cell ID {cell_id}, skipping...")
                            continue
                        
                        # Get cell boundary and ensure it's closed
                        cell_boundary = h3.cell_to_boundary(cell_id)
                        if cell_boundary[0] != cell_boundary[-1]:
                            cell_boundary = list(cell_boundary) + [cell_boundary[0]]
                        # Convert (lat, lon) to (lon, lat) for shapely
                        cell_boundary = [(lon, lat) for lat, lon in cell_boundary]
                        
                        # Create and validate cell polygon
                        cell_polygon = Polygon(cell_boundary)
                        if not cell_polygon.is_valid:
                            print(f"⚠️ Invalid cell geometry for {cell_id}, attempting to fix...")
                            cell_polygon = make_valid(cell_polygon)
                            if not cell_polygon.is_valid:
                                print(f"⚠️ Failed to fix cell geometry for {cell_id}, skipping...")
                                continue
                        
                        # Check for intersection with sub-territory
                        if sub_territory.intersects(cell_polygon):
                            intersection = sub_territory.intersection(cell_polygon)
                            if intersection.is_empty:
                                print(f"Cell {cell_id} discarded: empty intersection with sub-territory {sub_territory_idx}")
                                continue
                            
                            # Calculate areas in km² using latitude-dependent conversion
                            centroid = cell_polygon.centroid
                            lat_radians = math.radians(centroid.y)
                            deg_to_km = 111.32 * math.cos(lat_radians)  # Adjust for latitude
                            intersection_area_km2 = intersection.area * (1e6 / deg_to_km**2)
                            cell_area_km2 = cell_polygon.area * (1e6 / deg_to_km**2)
                            overlap_ratio = intersection_area_km2 / cell_area_km2 if cell_area_km2 > 0 else 0
                            
                            # Reference area from H3 library
                            cell_area_ref = h3.cell_area(cell_id, unit='km^2')  # Fixed unit from 'km2' to 'km^2'
                            # print(f"Cell {cell_id}: overlap_ratio={overlap_ratio:.3f}, "
                            #       f"intersection_area={intersection_area_km2:.3e} km², "
                            #       f"cell_area={cell_area_km2:.3e} km²")
                            
                            # Keep cells with significant overlap or area
                            if overlap_ratio > 0.01 or intersection_area_km2 > cell_area_ref * 0.5:
                                clipped_cells.add(cell_id)
                                cell_geometries[cell_id] = intersection
                            else:
                                print(f"Cell {cell_id} discarded: insufficient overlap (ratio={overlap_ratio:.3f})")
                        else:
                            print(f"Cell {cell_id} discarded: does not intersect sub-territory {sub_territory_idx}")
                    
                    except GEOSException as e:
                        print(f"⚠️ Geometry error for cell {cell_id}: {e}")
                        continue
                    except Exception as e:
                        print(f"⚠️ Unexpected error processing cell {cell_id}: {e}")
                        continue
                            
            except Exception as e:
                print(f"⚠️ Error processing sub-territory {sub_territory_idx}: {e}")
                continue
        
        # Calculate territory coverage
        territory_coverage = 0.0
        if territory_data['polygon'].area > 0 and cell_geometries:
            try:
                cell_union = unary_union(list(cell_geometries.values()))
                coverage_intersection = territory_data['polygon'].intersection(cell_union)
                territory_coverage = coverage_intersection.area / territory_data['polygon'].area
            except Exception as e:
                print(f"⚠️ Coverage calculation error for stock point {stock_point_id}: {e}")
        
        grid_results[stock_point_id] = {
            'h3_resolution': resolution,
            'h3_cells': all_h3_cells,
            'clipped_cells': clipped_cells,
            'cell_geometries': cell_geometries,
            'territory_coverage': territory_coverage
        }
        
        print(f"✅ Generated {len(clipped_cells)} clipped cells at resolution {resolution} "
              f"with coverage {territory_coverage:.3f}")
    
    print(f"🏁 Phase 2 complete: H3 grids generated for {len(grid_results)} territories")
    return grid_results


In [ ]:
import json
from shapely.geometry import Polygon
from shapely.validation import make_valid

def debug_territory_geometries(self, territories: Dict[str, Dict[str, Any]], stock_point_id: str, resolution: int = None):
    """
    Debugs territory and sub-territory geometries for a given stock_point_id.
    Prints WKT, GeoJSON, and analyzes H3 cell intersections.
    
    Args:
        territories: Dictionary from define_territories.
        stock_point_id: ID of the territory to debug.
        resolution: H3 resolution to test (optional, uses _calculate_adaptive_resolution if None).
    """
    if stock_point_id not in territories:
        print(f"❌ Stock point {stock_point_id} not found in territories")
        return
    
    territory_data = territories[stock_point_id]
    print(f"\n🔍 Debugging territory for stock point {stock_point_id}")
    
    # Print territory details
    print(f"Is contiguous: {territory_data['is_contiguous']}")
    print(f"LGA count: {territory_data['lga_count']}")
    print(f"Total area (km²): {territory_data['total_area_km2']:.3f}")
    print(f"Territory polygon (WKT): {territory_data['polygon'].wkt}")
    
    # Print sub-territory details
    print(f"Sub-territories count: {len(territory_data['sub_territories'])}")
    for i, sub_territory in enumerate(territory_data['sub_territories']):
        area_km2 = sub_territory.area * 1e6 / 111.32**2
        print(f"Sub-territory {i} (WKT): {sub_territory.wkt}")
        print(f"Sub-territory {i} area: {area_km2:.3e} km²")
        print(f"Sub-territory {i} (GeoJSON): {json.dumps(sub_territory.__geo_interface__)}")
    
    # Test H3 cell generation and intersections
    resolution = resolution if resolution is not None else self._calculate_adaptive_resolution(territory_data['lga_ids'])
    print(f"Testing with resolution: {resolution}")
    
    for i, sub_territory in enumerate(territory_data['sub_territories']):
        if not sub_territory.is_valid:
            print(f"⚠️ Invalid sub-territory {i} geometry, fixing...")
            sub_territory = make_valid(sub_territory)
        
        try:
            h3_shape = h3.geo_to_h3shape(sub_territory.__geo_interface__)
            sub_cells = set(h3.polygon_to_cells(h3_shape, resolution))
            print(f"Sub-territory {i} generated {len(sub_cells)} H3 cells")
            
            intersected_cells = 0
            for cell_id in sub_cells:
                try:
                    cell_boundary = h3.cell_to_boundary(cell_id)
                    if cell_boundary[0] != cell_boundary[-1]:
                        cell_boundary = list(cell_boundary) + [cell_boundary[0]]
                    cell_boundary = [(lon, lat) for lat, lon in cell_boundary]
                    cell_polygon = Polygon(cell_boundary)
                    if not cell_polygon.is_valid:
                        cell_polygon = make_valid(cell_polygon)
                    
                    if sub_territory.intersects(cell_polygon):
                        intersected_cells += 1
                        intersection = sub_territory.intersection(cell_polygon)
                        if not intersection.is_empty:
                            intersection_area_km2 = intersection.area * 1e6 / 111.32**2
                            cell_area_km2 = cell_polygon.area * 1e6 / 111.32**2
                            overlap_ratio = intersection_area_km2 / cell_area_km2 if cell_area_km2 > 0 else 0
                            print(f"Cell {cell_id}: intersects sub-territory {i}, overlap_ratio={overlap_ratio:.3f}")
                        else:
                            print(f"Cell {cell_id}: intersects but empty intersection")
                    else:
                        print(f"Cell {cell_id} does not intersect sub-territory {i}")
                except Exception as e:
                    print(f"⚠️ Error processing cell {cell_id} for sub-territory {i}: {e}")
            print(f"Sub-territory {i} intersected {intersected_cells}/{len(sub_cells)} cells")
        except Exception as e:
            print(f"⚠️ Error processing sub-territory {i}: {e}")
    
    print(f"🔍 Debugging complete for {stock_point_id}")

In [ ]:
h3.__version__

In [ ]:
## Phase 2
cw_grid_results  = generate_h3_grids(clusterer, territories=cw_terriroty)
# debug_territory_geometries(clusterer, territories=territories,stock_point_id=str(stock_point_id),resolution=8)

In [ ]:
cw_grid_results['1647113'].keys()

#### Keep

In [ ]:
print(cw_grid_results[str(stock_point_id)].keys())
print('h3_resolution', cw_grid_results[str(stock_point_id)]['h3_resolution']) # H3 resolution
print('territory_coverage', cw_grid_results[str(stock_point_id)]['territory_coverage']) # territory_coverage
# print('territory_coverage', cw_grid_results[str(stock_point_id)]['territory_coverage']) # H3 resolution
# cw_terriroty_polygon = cw_terriroty[str(stock_point_id)]['polygon'] # territory polygon
# print('Territory polygon area:', cw_terriroty_polygon.area) # territory polygon area
# print('Territory polygon is valid:', cw_terriroty_polygon.is_valid) #
# cw_terriroty_h3_cells = cw_grid_results[str(stock_point_id)]['h3_cells'] # H3 cells
# print('Number of H3 cells:', len(cw_terriroty_h3_cells)) # Number of H3 cells
# cw_terriroty_clipped_cells = cw_grid_results[str(stock_point_id)]['clipped_cells'] # Clipped cells
# print('Number of clipped cells:', len(cw_terriroty_clipped_cells)) # Number of H3 cells
# cw_terriroty_cell_geometries = cw_grid_results[str(stock_point_id)]['cell_geometries'] # Clipped cells

# # type(cw_terriroty_polygon) # shapely.geometry.polygon.Polygon
# # set([type(cell) for cell in cw_terriroty_h3_cells]) # List of H3 cell IDs)]
# # set([type(cell) for cell in cw_terriroty_clipped_cells]) # List of H3 cell IDs)]
# # cw_terriroty_cell_geometries

In [ ]:
# plot_territory(cw_terriroty[str(stock_point_id)]['polygon'], stock_point_id=1647113)
# territory_gdf_to_plot = gpd.GeoDataFrame(geometry=territory_polygons, crs="EPSG:4326")
cw_terriroty[str(stock_point_id)]['polygon'] 
 

#### Phase 3

In [ ]:
## Phase 3: 3-Tier Customer Assignment Workflow
def assign_customers_to_clusters(self, grid_results: Dict[str, Dict[str, Any]]) -> Dict[str, gpd.GeoDataFrame]:
    """
    Phase 3: 3-Tier Customer Assignment Workflow
    
    Assigns customers to H3 clusters using hierarchical confidence levels:
    Tier 1: H3 cell inclusion (confidence: 1.0)
    Tier 2: Point-in-polygon (confidence: 0.8) 
    Tier 3: Manual review (confidence: 0.0)
    
    Returns:
        Dict[stock_point_id, assignments_gdf with columns:
            ['customer_id', 'cluster_id', 'h3_cell_id', 'assignment_confidence', 'assignment_tier']]
    """
    print("👥 Phase 3: Starting customer assignment...")
    
    all_assignments = {}
    assignment_stats = {'tier1': 0, 'tier2': 0, 'tier3': 0}
    
    for stock_point_id, grid_data in grid_results.items():
        print(f"Assigning customers for stock point {stock_point_id}...")
        
        # Get customers for this stock point
        customers_sp = self.customers[self.customers['stock_point_id'] == int(stock_point_id)].copy()
        print(f"Number of Customers {len(customers_sp)}")
        
        if customers_sp.empty:
            all_assignments[stock_point_id] = gpd.GeoDataFrame()
            continue
        
        assignments = []
        resolution = grid_data['h3_resolution']
        clipped_cells = grid_data['clipped_cells']
        cell_geometries = grid_data['cell_geometries']
        
        for _, customer in customers_sp.iterrows():
            lat, lng = customer.geometry.y, customer.geometry.x
            customer_id = customer['customer_id']
            
            # Tier 1: H3 cell inclusion
            try:
                h3_cell = h3.latlng_to_cell(lat, lng, resolution)
                if h3_cell in clipped_cells:
                    assignments.append({
                        'customer_id': customer_id,
                        'cluster_id': h3_cell,
                        'h3_cell_id': h3_cell,
                        'assignment_confidence': 1.0,
                        'assignment_tier': 'h3_inclusion'
                    })
                    assignment_stats['tier1'] += 1
                    continue
            except:
                pass
            
            # Tier 2: Point-in-polygon check
            assigned = False
            customer_point = customer.geometry
            
            for cell_id, cell_geom in cell_geometries.items():
                if cell_geom.contains(customer_point):
                    assignments.append({
                        'customer_id': customer_id,
                        'cluster_id': cell_id,
                        'h3_cell_id': cell_id,
                        'assignment_confidence': 0.8,
                        'assignment_tier': 'point_in_polygon'
                    })
                    assignment_stats['tier2'] += 1
                    assigned = True
                    break
            
            if assigned:
                continue
                
            # Tier 3: Manual review
            assignments.append({
                'customer_id': customer_id,
                'cluster_id': None,
                'h3_cell_id': None,
                'assignment_confidence': 0.0,
                'assignment_tier': 'manual_review'
            })
            assignment_stats['tier3'] += 1
        
        # Create GeoDataFrame
        assignments_df = pd.DataFrame(assignments)
        assignments_gdf = gpd.GeoDataFrame(
            assignments_df,
            geometry=[customers_sp[customers_sp['customer_id'] == cid].geometry.iloc[0] 
                     for cid in assignments_df['customer_id']]
        )
        
        all_assignments[stock_point_id] = assignments_gdf
        print(f"✅ Assigned {len(assignments_df)} customers")
    
    # Log statistics
    total = sum(assignment_stats.values())
    if total > 0:
        print(f"📊 Assignment stats - Tier1: {assignment_stats['tier1']/total:.1%}, "
              f"Tier2: {assignment_stats['tier2']/total:.1%}, "
              f"Tier3: {assignment_stats['tier3']/total:.1%}")
    
    print(f"🏁 Phase 3 complete: {total} customers assigned")
    return all_assignments

In [ ]:
## Testing phase 3
assignments = assign_customers_to_clusters(clusterer, grid_results=cw_grid_results)

In [ ]:


# Print summary stats about cluster_id and and h3_cell_id
def print_assignment_summary(assignments: gpd.GeoDataFrame):
    """
    Prints summary statistics for customer assignments.
    
    Args:
        assignments: GeoDataFrame with customer assignments.
    """
    if assignments.empty:
        print("No assignments to summarize.")
        return
    
    total_customers = len(assignments)
    tier_counts = assignments['assignment_tier'].value_counts()
    
    print(f"Total Customers Assigned: {total_customers}")
    print("Assignment Tiers:")
    for tier, count in tier_counts.items():
        print(f"  {tier}: {count} ({count / total_customers:.1%})")
    
    # Cluster ID and H3 Cell ID stats
    cluster_counts = assignments['cluster_id'].value_counts()
    h3_cell_counts = assignments['h3_cell_id'].value_counts()
    
    print(f"\nUnique Clusters Assigned: {len(cluster_counts)}")
    print(f"Unique H3 Cells Assigned: {len(h3_cell_counts)}")
    print(f"Most Common Cluster ID: {cluster_counts.idxmax()} ({cluster_counts.max()})")
    print(f"Most Common H3 Cell ID: {h3_cell_counts.idxmax()} ({h3_cell_counts.max()})")

    # Print top 3 clusters and number of customers assigned
    print("\nTop 3 Clusters by Customer Count:")
    top_clusters = cluster_counts.head(3)
    for cluster_id, count in top_clusters.items():
        print(f"  Cluster ID {cluster_id}: {count} customers")
        
    # Print bottom 10 clusters and number of customers assigned
    print("\nBottom 3 Clusters by Customer Count:")
    bottom_clusters = cluster_counts.tail(3)
    for cluster_id, count in bottom_clusters.items():
        print(f"  Cluster ID {cluster_id}: {count} customers")

print_assignment_summary(assignments = assignments['1647113'])

#### Phase 4

In [ ]:
## Phase 4: Cluster Optimization - Splitting and Merging
def optimize_clusters_(self, assignments: Dict[str, gpd.GeoDataFrame], 
                     grid_results: Dict[str, Dict[str, Any]]) -> Dict[str, pd.DataFrame]:
    """
    Phase 4: Cluster Optimization - Splitting and Merging
    
    Splits high-density clusters and merges sparse clusters based on customer counts.
    Urban splitting: >200 customers → split to higher resolution
    Rural merging: <10 customers → merge with neighbors
    
    Returns:
        Dict[stock_point_id, optimized_clusters_df with columns:
            ['cluster_id', 'h3_resolution', 'h3_cells', 'customer_count', 'parent_cluster_id']]
    """
    print("⚙️ Phase 4: Starting cluster optimization...")
    
    optimized_results = {}
    
    for stock_point_id, assignments_gdf in assignments.items():
        print(f"Optimizing clusters for stock point {stock_point_id}...")
        
        if assignments_gdf.empty:
            optimized_results[stock_point_id] = pd.DataFrame()
            continue
        
        grid_data = grid_results[stock_point_id]
        base_resolution = grid_data['h3_resolution']
        
        # Count customers per cluster
        cluster_counts = assignments_gdf[assignments_gdf['cluster_id'].notna()].groupby('cluster_id').size().reset_index(name='customer_count')
        
        # Initialize optimized clusters
        optimized_clusters = []
        processed_clusters = set()
        
        # Step 1: Split high-density clusters
        high_density = cluster_counts[cluster_counts['customer_count'] > CUSTOMER_DENSITY_THRESHOLD_HIGH]
        
        for _, row in high_density.iterrows():
            parent_cell = row['cluster_id']
            customer_count = row['customer_count']
            
            if base_resolution < MAX_H3_RESOLUTION_URBAN:
                # Split into child cells
                child_resolution = base_resolution + 1
                child_cells = list(h3.cell_to_children(parent_cell, child_resolution))
                
                # Get customers in this cluster
                cluster_customers = assignments_gdf[assignments_gdf['cluster_id'] == parent_cell]
                
                # Reassign customers to child cells
                for child_cell in child_cells:
                    child_customers = []
                    for _, customer in cluster_customers.iterrows():
                        lat, lng = customer.geometry.y, customer.geometry.x
                        customer_h3_cell = h3.latlng_to_cell(lat, lng, child_resolution)
                        if customer_h3_cell == child_cell:
                            child_customers.append(customer)
                    
                    if child_customers:
                        optimized_clusters.append({
                            'cluster_id': child_cell,
                            'h3_resolution': child_resolution,
                            'h3_cells': [child_cell],
                            'customer_count': len(child_customers),
                            'parent_cluster_id': parent_cell
                        })
                
                processed_clusters.add(parent_cell)
                print(f"🔄 Split cluster {parent_cell} ({customer_count} customers) into {len(child_cells)} child clusters")
        
        # Step 2: Keep standard density clusters
        standard_density = cluster_counts[
            (cluster_counts['customer_count'] <= CUSTOMER_DENSITY_THRESHOLD_HIGH) &
            (cluster_counts['customer_count'] >= CUSTOMER_DENSITY_THRESHOLD_LOW) &
            (~cluster_counts['cluster_id'].isin(processed_clusters))
        ]
        
        for _, row in standard_density.iterrows():
            optimized_clusters.append({
                'cluster_id': row['cluster_id'],
                'h3_resolution': base_resolution,
                'h3_cells': [row['cluster_id']],
                'customer_count': row['customer_count'],
                'parent_cluster_id': None
            })
            processed_clusters.add(row['cluster_id'])
        
        # Step 3: Merge low-density clusters
        low_density = cluster_counts[
            (cluster_counts['customer_count'] < CUSTOMER_DENSITY_THRESHOLD_LOW) &
            (~cluster_counts['cluster_id'].isin(processed_clusters))
        ]
        
        merge_map = {}
        territory_cells = grid_data['clipped_cells']
        
        for _, row in low_density.sort_values('customer_count').iterrows():
            sparse_cell = row['cluster_id']
            
            if sparse_cell in merge_map:
                continue
                
            # Find neighboring cells within territory
            neighbors = set(h3.grid_disk(sparse_cell, MERGE_SEARCH_RADIUS)).intersection(territory_cells)
            neighbor_counts = cluster_counts[cluster_counts['cluster_id'].isin(neighbors)]
            
            # Find best merge target (highest customer count, not already processed)
            valid_neighbors = neighbor_counts[
                (~neighbor_counts['cluster_id'].isin(processed_clusters)) &
                (neighbor_counts['customer_count'] >= CUSTOMER_DENSITY_THRESHOLD_LOW)
            ]
            
            if not valid_neighbors.empty:
                target = valid_neighbors.loc[valid_neighbors['customer_count'].idxmax()]
                merge_map[sparse_cell] = target['cluster_id']
                print(f"🔗 Merging sparse cluster {sparse_cell} ({row['customer_count']} customers) → {target['cluster_id']}")
        
        # Apply merges
        for sparse_cell, target_cell in merge_map.items():
            # Find or create target cluster
            target_cluster = next((c for c in optimized_clusters if c['cluster_id'] == target_cell), None)
            
            if not target_cluster:
                target_count = cluster_counts[cluster_counts['cluster_id'] == target_cell]['customer_count'].iloc[0]
                target_cluster = {
                    'cluster_id': target_cell,
                    'h3_resolution': base_resolution,
                    'h3_cells': [target_cell],
                    'customer_count': target_count,
                    'parent_cluster_id': None
                }
                optimized_clusters.append(target_cluster)
                processed_clusters.add(target_cell)
            
            # Add sparse cell to target
            sparse_count = cluster_counts[cluster_counts['cluster_id'] == sparse_cell]['customer_count'].iloc[0]
            target_cluster['h3_cells'].append(sparse_cell)
            target_cluster['customer_count'] += sparse_count
            processed_clusters.add(sparse_cell)
        
        # Convert to DataFrame
        optimized_df = pd.DataFrame(optimized_clusters)
        optimized_df['stock_point_id'] = stock_point_id
        
        optimized_results[stock_point_id] = optimized_df
        print(f"✅ Optimized to {len(optimized_df)} clusters (split: {len(high_density)}, merged: {len(merge_map)})")
    
    print(f"🏁 Phase 4 complete: Cluster optimization finished")
    return optimized_results


def optimize_clusters(self, assignments: Dict[str, gpd.GeoDataFrame], 
                     grid_results: Dict[str, Dict[str, Any]]) -> Dict[str, pd.DataFrame]:
    """
    Phase 4: Cluster Optimization - Splitting and Merging
    
    Splits high-density clusters and merges sparse clusters based on customer counts.
    Urban splitting: >200 customers → split to higher resolution
    Rural merging: <10 customers → merge with neighbors
    Retains unmerged low-density clusters to prevent customer loss.
    
    Returns:
        Dict[stock_point_id, optimized_clusters_df with columns:
            ['cluster_id', 'h3_resolution', 'h3_cells', 'customer_count', 'parent_cluster_id']]
    """
    print("⚙️ Phase 4: Starting cluster optimization...")
    
    optimized_results = {}
    
    for stock_point_id, assignments_gdf in assignments.items():
        print(f"Optimizing clusters for stock point {stock_point_id}...")
        
        if assignments_gdf.empty:
            optimized_results[stock_point_id] = pd.DataFrame()
            continue
        
        grid_data = grid_results[stock_point_id]
        base_resolution = grid_data['h3_resolution']
        
        # Count customers per cluster
        cluster_counts = assignments_gdf[assignments_gdf['cluster_id'].notna()].groupby('cluster_id').size().reset_index(name='customer_count')
        
        # Initialize optimized clusters
        optimized_clusters = []
        processed_clusters = set()
        
        # Step 1: Split high-density clusters
        high_density = cluster_counts[cluster_counts['customer_count'] > CUSTOMER_DENSITY_THRESHOLD_HIGH]
        
        for _, row in high_density.iterrows():
            parent_cell = row['cluster_id']
            customer_count = row['customer_count']
            
            if base_resolution < MAX_H3_RESOLUTION_URBAN:
                # Split into child cells
                child_resolution = base_resolution + 1
                child_cells = list(h3.cell_to_children(parent_cell, child_resolution))
                
                # Get customers in this cluster
                cluster_customers = assignments_gdf[assignments_gdf['cluster_id'] == parent_cell]
                
                # Reassign customers to child cells
                for child_cell in child_cells:
                    child_customers = []
                    for _, customer in cluster_customers.iterrows():
                        lat, lng = customer.geometry.y, customer.geometry.x
                        customer_h3_cell = h3.latlng_to_cell(lat, lng, child_resolution)
                        if customer_h3_cell == child_cell:
                            child_customers.append(customer)
                    
                    if child_customers:
                        optimized_clusters.append({
                            'cluster_id': child_cell,
                            'h3_resolution': child_resolution,
                            'h3_cells': [child_cell],
                            'customer_count': len(child_customers),
                            'parent_cluster_id': parent_cell
                        })
                
                processed_clusters.add(parent_cell)
                print(f"🔄 Split cluster {parent_cell} ({customer_count} customers) into {len(child_cells)} child clusters")
        
        # Step 2: Keep standard-density clusters
        standard_density = cluster_counts[
            (cluster_counts['customer_count'] <= CUSTOMER_DENSITY_THRESHOLD_HIGH) &
            (cluster_counts['customer_count'] >= CUSTOMER_DENSITY_THRESHOLD_LOW) &
            (~cluster_counts['cluster_id'].isin(processed_clusters))
        ]
        
        for _, row in standard_density.iterrows():
            optimized_clusters.append({
                'cluster_id': row['cluster_id'],
                'h3_resolution': base_resolution,
                'h3_cells': [row['cluster_id']],
                'customer_count': row['customer_count'],
                'parent_cluster_id': None
            })
            processed_clusters.add(row['cluster_id'])
        
        # Step 3: Merge low-density clusters
        low_density = cluster_counts[
            (cluster_counts['customer_count'] < CUSTOMER_DENSITY_THRESHOLD_LOW) &
            (~cluster_counts['cluster_id'].isin(processed_clusters))
        ]
        
        merge_map = {}
        territory_cells = grid_data['clipped_cells']
        
        for _, row in low_density.sort_values('customer_count').iterrows():
            sparse_cell = row['cluster_id']
            
            if sparse_cell in merge_map:
                continue
                
            # Find neighboring cells within territory
            neighbors = set(h3.grid_disk(sparse_cell, MERGE_SEARCH_RADIUS)).intersection(territory_cells)
            neighbor_counts = cluster_counts[cluster_counts['cluster_id'].isin(neighbors)]
            
            # Find best merge target (highest customer count, not already processed)
            valid_neighbors = neighbor_counts[
                (~neighbor_counts['cluster_id'].isin(processed_clusters)) &
                (neighbor_counts['customer_count'] >= CUSTOMER_DENSITY_THRESHOLD_LOW)
            ]
            
            if not valid_neighbors.empty:
                target = valid_neighbors.loc[valid_neighbors['customer_count'].idxmax()]
                merge_map[sparse_cell] = target['cluster_id']
                # print(f"🔗 Merging sparse cluster {sparse_cell} ({row['customer_count']} customers) → {target['cluster_id']}")
            else:
                # Retain unmerged sparse cluster to prevent customer loss
                optimized_clusters.append({
                    'cluster_id': sparse_cell,
                    'h3_resolution': base_resolution,
                    'h3_cells': [sparse_cell],
                    'customer_count': row['customer_count'],
                    'parent_cluster_id': None
                })
                processed_clusters.add(sparse_cell)
                # print(f"⚠️ Sparse cluster {sparse_cell} ({row['customer_count']} customers) not merged, retained as is")
        
        # Step 4: Apply merges
        for sparse_cell, target_cell in merge_map.items():
            # Find or create target cluster
            target_cluster = next((c for c in optimized_clusters if c['cluster_id'] == target_cell), None)
            
            if not target_cluster:
                target_count = cluster_counts[cluster_counts['cluster_id'] == target_cell]['customer_count'].iloc[0]
                target_cluster = {
                    'cluster_id': target_cell,
                    'h3_resolution': base_resolution,
                    'h3_cells': [target_cell],
                    'customer_count': target_count,
                    'parent_cluster_id': None
                }
                optimized_clusters.append(target_cluster)
                processed_clusters.add(target_cell)
            
            # Add sparse cell to target
            sparse_count = cluster_counts[cluster_counts['cluster_id'] == sparse_cell]['customer_count'].iloc[0]
            target_cluster['h3_cells'].append(sparse_cell)
            target_cluster['customer_count'] += sparse_count
            processed_clusters.add(sparse_cell)
        
        # Convert to DataFrame
        optimized_df = pd.DataFrame(optimized_clusters)
        optimized_df['stock_point_id'] = stock_point_id
        
        optimized_results[stock_point_id] = optimized_df
        print(f"✅ Optimized to {len(optimized_df)} clusters (split: {len(high_density)}, merged: {len(merge_map)}, retained sparse: {len(low_density) - len(merge_map)})")
    
    print(f"🏁 Phase 4 complete: Cluster optimization finished")
    return optimized_results



In [ ]:
CUSTOMER_DENSITY_THRESHOLD_HIGH

In [ ]:
### Test Phase 4
optimized_clusters = optimize_clusters(clusterer, assignments=assignments, grid_results=cw_grid_results)

In [ ]:
optimized_clusters['1647113'].keys()

In [ ]:

print(f"Total Customers in Stock point {stock_point_id}: {len(assignments[str(stock_point_id)])}")
print(f"Total assigned clusters for stock point {assignments[str(stock_point_id)].query('~cluster_id.isnull()').cluster_id.nunique()}")
print(f"Total customers in assigned clusters: {assignments[str(stock_point_id)].query('~cluster_id.isnull()').customer_id.count()}")

print(f"Total clusters optimized for stock point {stock_point_id}: {len(optimized_clusters[str(stock_point_id)])}")
print(f"Total customers in optimized clusters: {optimized_clusters[str(stock_point_id)].customer_count.sum()}")


In [ ]:
# How many cluster_id is None
print(assignments[str(stock_point_id)].query("~cluster_id.isnull()").customer_id.count())

In [ ]:
optimized_clusters[str(stock_point_id)].customer_count.sum()#.tail()

#### Main Class

#### Utils

In [ ]:
# plot_territory(territory_polygons, stock_point_id=1647113)

In [ ]:
# plot_territory_and_h3_overlay(
#     territory_polygons=territory_polygons,
#     territory_cells = territory_cells,
#     title = "Territory with H3 Grid Overlay",
#     h3_resolution = None # Optional: for display purposes
# )

## WILD RUN

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon, MultiPolygon
from shapely.ops import unary_union # Added for territory definition
import h3

class H3SpatialClusterer:
    """
    A class to perform spatial clustering and analysis using H3 indexing,
    to define and validate geographical territories for stock points,
    and to generate H3 grids for these territories.
    """

    def __init__(self, customer_data: gpd.GeoDataFrame,
                 stock_point_dim: pd.DataFrame,
                 stock_point_lga_map: pd.DataFrame,
                 lga_gdf: gpd.GeoDataFrame,
                 h3_resolution: int = 7):
        """
        Initializes the H3SpatialClusterer with the necessary dataframes and H3 resolution.

        Args:
            customer_data (gpd.GeoDataFrame): DataFrame containing customer
                                              information including geometry.
                                              Expected columns: 'customer_id',
                                              'stock_point_id', 'longitude',
                                              'latitude', 'geometry'.
            stock_point_dim (pd.DataFrame): DataFrame with stock point dimensions.
                                            Expected columns: 'stock_point_id',
                                            'stock_point_name', 'latitude',
                                            'longitude'.
            stock_point_lga_map (pd.DataFrame): DataFrame mapping stock points to LGAs.
                                                Expected columns: 'stock_point_id',
                                                'lga_id'.
            lga_gdf (gpd.GeoDataFrame): GeoDataFrame containing LGA geographical data.
                                        Expected columns: 'lga_id', 'lga_name',
                                        'geometry', 'area_km2'.
            h3_resolution (int): The H3 resolution to use for indexing. Default is 7.
        """
        self.customer_data = customer_data
        self.stock_point_dim = stock_point_dim
        self.stock_point_lga_map = stock_point_lga_map
        self.lga_gdf = lga_gdf
        self.h3_resolution = h3_resolution
        self.merged_h3_data = None
        self.territories = {} # Stores result of Phase 1: Territory Definition
        self.h3_grids = {} # Stores result of Phase 2: H3 Grid Generation

        print(f"H3SpatialClusterer initialized with provided dataframes and H3 resolution {self.h3_resolution}.")

    def define_territories(self) -> dict:
        """
        Phase 1: Territory Definition and Validation

        Creates boundary-constrained territories for each stock point using LGA geometries.
        Handles non-contiguous territories and validates geometric integrity.

        Returns:
            Dict[stock_point_id, {
                'polygon': Union[Polygon, MultiPolygon],
                'lga_ids': List[str],
                'is_contiguous': bool,
                'sub_territories': List[Polygon],
                'total_area_km2': float,
                'territory_version': str,
                'lga_count': int,
                'validation_status': str
            }]
        """
        print("🗺️ Phase 1: Starting territory definition...")

        self.territories = {} # Initialize or clear territories
        
        # Get all unique stock points from the mapping
        stock_points = self.stock_point_lga_map['stock_point_id'].unique()

        for stock_point_id in stock_points:
            print(f"Processing territory for stock point {stock_point_id}...")

            # Get LGA IDs for this stock point
            lga_ids = self.stock_point_lga_map[
                self.stock_point_lga_map['stock_point_id'] == stock_point_id
            ]['lga_id'].tolist()

            if not lga_ids:
                print(f"⚠️ Warning: No LGAs found for stock_point_id {stock_point_id}")
                continue

            # Get LGA geometries using self.lga_gdf
            territory_lgas = self.lga_gdf[self.lga_gdf['lga_id'].isin(lga_ids)].copy()

            if territory_lgas.empty:
                print(f"⚠️ Warning: No LGA geometries found for stock_point_id {stock_point_id}")
                continue

            # Validate individual LGA geometries
            validated_geometries = []
            for idx, lga in territory_lgas.iterrows():
                geom = lga['geometry']
                if not geom.is_valid:
                    print(f"🔧 Fixing invalid geometry for LGA {lga['lga_id']}")
                    geom = geom.buffer(0)  # Fix self-intersections
                validated_geometries.append(geom)

            # Create unified territory
            if len(validated_geometries) == 1:
                unified_territory = validated_geometries[0]
            else:
                # Union all LGA geometries
                unified_territory = unary_union(validated_geometries)

            # Final validation of unified territory
            if not unified_territory.is_valid:
                print(f"🔧 Fixing unified territory geometry for stock point {stock_point_id}")
                unified_territory = unified_territory.buffer(0)

            # Handle non-contiguous territories
            is_contiguous = isinstance(unified_territory, Polygon)
            sub_territories = []

            if isinstance(unified_territory, Polygon):
                sub_territories = [unified_territory]
            elif isinstance(unified_territory, MultiPolygon):
                sub_territories = list(unified_territory.geoms)
                print(f"📍 Non-contiguous territory detected for {stock_point_id}: {len(sub_territories)} sub-territories")
            else:
                print(f"⚠️ Unexpected geometry type for stock point {stock_point_id}: {type(unified_territory)}")
                self.territories[str(stock_point_id)] = {
                    'polygon': None,
                    'lga_ids': lga_ids,
                    'is_contiguous': False,
                    'sub_territories': [],
                    'total_area_km2': 0,
                    'territory_version': 'v1.2',
                    'lga_count': len(lga_ids),
                    'validation_status': 'invalid_geometry'
                }
                continue # Skip to next stock point if geometry is unexpected

            # Calculate total area
            total_area_km2 = territory_lgas['area_km2'].sum() if 'area_km2' in territory_lgas.columns else 0
            if total_area_km2 == 0:
                print(f"⚠️ Warning: 'area_km2' column not found or sum is zero for stock point {stock_point_id}. Area set to 0.")

            # Store territory information
            self.territories[str(stock_point_id)] = {
                'polygon': unified_territory,
                'lga_ids': lga_ids,
                'is_contiguous': is_contiguous,
                'sub_territories': sub_territories,
                'total_area_km2': total_area_km2,
                'territory_version': 'v1.2',
                'lga_count': len(lga_ids),
                'validation_status': 'valid'
            }

            print(f"✅ Territory defined for {stock_point_id}: {len(lga_ids)} LGAs, {len(sub_territories)} sub-territories, {total_area_km2:.1f} km²")

        print(f"🏁 Phase 1 complete: {len(self.territories)} territories defined")
        return self.territories

    def _calculate_adaptive_resolution(self, lga_ids: list) -> int:
        """
        Calculates an adaptive H3 resolution for a territory based on its LGAs.
        This is a simplified heuristic based on total area and number of LGAs.
        In a real-world scenario, this would be refined with actual
        population/area density data.

        Args:
            lga_ids (list): List of LGA IDs belonging to the territory.

        Returns:
            int: The calculated H3 resolution.
        """
        if not lga_ids:
            return self.h3_resolution # Default resolution if no LGAs

        # Get relevant LGA areas
        relevant_lgas = self.lga_gdf[self.lga_gdf['lga_id'].isin(lga_ids)]
        if relevant_lgas.empty or 'area_km2' not in relevant_lgas.columns:
            print("Warning: Could not get area_km2 for adaptive resolution. Using default.")
            return self.h3_resolution

        total_area = relevant_lgas['area_km2'].sum()
        num_lgas = len(lga_ids)

        # Simple heuristic:
        # Smaller total area / fewer LGAs -> higher resolution (finer grid)
        # Larger total area / more LGAs -> lower resolution (coarser grid)
        # These thresholds are arbitrary and should be tuned based on actual data characteristics.
        if total_area < 150 and num_lgas <= 2:
            return min(self.h3_resolution + 1, 10) # Finer resolution (e.g., 7 -> 8)
        elif total_area > 300 or num_lgas >= 4:
            return max(self.h3_resolution - 1, 4) # Coarser resolution (e.g., 7 -> 6)
        else:
            return self.h3_resolution # Default resolution

    def generate_h3_grids(self, territories: dict) -> dict:
        """
        Phase 2: H3 Grid Generation with Adaptive Resolution

        Generates boundary-clipped H3 hexagons for each territory with density-based resolution.

        Args:
            territories (Dict[str, Dict[str, Any]]): The output from define_territories.

        Returns:
            Dict[stock_point_id, {
                'h3_resolution': int,
                'h3_cells': Set[str],
                'clipped_cells': Set[str],
                'cell_geometries': Dict[str, Polygon],
                'territory_coverage': float
            }]
        """
        print("🔶 Phase 2: Starting H3 grid generation...")
        
        self.h3_grids = {} # Initialize or clear grid results
        
        # Define a projected CRS for accurate area calculations (e.g., World Equidistant Cylindrical)
        # EPSG:6933 uses meters. Convert to km for consistency with area_km2.
        PROJECTED_CRS = "EPSG:6933" 
        
        for stock_point_id, territory_data in territories.items():
            print(f"Generating H3 grid for stock point {stock_point_id}...")
            
            # Calculate adaptive resolution
            resolution = self._calculate_adaptive_resolution(territory_data['lga_ids'])
            
            all_h3_cells = set()
            clipped_cells = set()
            cell_geometries = {}
            
            # Reproject the main territory polygon once for area calculation
            # Ensure territory_data['polygon'] is a GeoSeries or GeoDataFrame for .to_crs()
            # If it's a raw shapely geometry, create a temporary GeoSeries
            if isinstance(territory_data['polygon'], (Polygon, MultiPolygon)):
                territory_polygon_gs = gpd.GeoSeries([territory_data['polygon']], crs="EPSG:4326")
                projected_territory_polygon = territory_polygon_gs.to_crs(PROJECTED_CRS).iloc[0]
            else:
                print(f"⚠️ Skipping H3 grid for {stock_point_id}: Invalid territory polygon type.")
                continue


            for sub_territory in territory_data['sub_territories']:
                # Convert shapely polygon to H3 polyfill format: [(lat, lon), ...] for exterior, then holes
                # h3.polygon_to_cells expects (lat, lon) for the tuple representation of a polygon.
                # Shapely's .coords typically return (lon, lat), so we reverse them.
                if sub_territory.geom_type == 'Polygon':
                    exterior_coords = [(coord[1], coord[0]) for coord in sub_territory.exterior.coords]
                    interior_coords = [[(coord[1], coord[0]) for coord in interior.coords] for interior in sub_territory.interiors]
                    h3_polygon_format = (exterior_coords, interior_coords)
                elif sub_territory.geom_type == 'MultiPolygon':
                    # For MultiPolygon, h3.polygon_to_cells expects a single polygon at a time.
                    # We iterate through sub_territories which are already single polygons (or MultiPolygon components).
                    # This case should ideally not be hit if sub_territories are always single Polygons.
                    print(f"Warning: MultiPolygon found in sub_territories for {stock_point_id}. Processing as single polygons.")
                    exterior_coords = [(coord[1], coord[0]) for coord in sub_territory.exterior.coords]
                    interior_coords = [[(coord[1], coord[0]) for coord in interior.coords] for interior in sub_territory.interiors]
                    h3_polygon_format = (exterior_coords, interior_coords)
                else:
                    print(f"⚠️ Skipping sub-territory due to unsupported geometry type: {sub_territory.geom_type}")
                    continue

                try:
                    sub_cells = set(h3.polygon_to_cells(h3_polygon_format, resolution))
                    all_h3_cells.update(sub_cells)
                    
                    # Clip cells to territory boundaries
                    for cell_id in sub_cells:
                        # h3.h3_to_geo_boundary with geo_json=True returns (lon, lat) pairs
                        cell_boundary_lon_lat = h3.h3_to_geo_boundary(cell_id, geo_json=True)
                        cell_polygon_wgs84 = Polygon(cell_boundary_lon_lat) # Original CRS polygon

                        # Reproject both the sub_territory and the cell_polygon to the projected CRS
                        # for accurate area intersection calculation
                        projected_sub_territory = gpd.GeoSeries([sub_territory], crs="EPSG:4326").to_crs(PROJECTED_CRS).iloc[0]
                        projected_cell_polygon = gpd.GeoSeries([cell_polygon_wgs84], crs="EPSG:4326").to_crs(PROJECTED_CRS).iloc[0]
                        
                        if projected_sub_territory.intersects(projected_cell_polygon):
                            intersection = projected_sub_territory.intersection(projected_cell_polygon)
                            
                            # Convert area to square kilometers for meaningful comparison
                            intersection_area_km2 = intersection.area / (1000 * 1000) # Area in m^2, convert to km^2
                            cell_area_km2 = projected_cell_polygon.area / (1000 * 1000) # Area in m^2, convert to km^2

                            # Keep cells with >10% overlap (now based on km^2)
                            if cell_area_km2 > 0 and intersection_area_km2 > (cell_area_km2 * 0.1):
                                clipped_cells.add(cell_id)
                                # Store the intersection geometry in the original WGS84 CRS
                                cell_geometries[cell_id] = intersection.to_crs("EPSG:4326")
                                
                except Exception as e:
                    print(f"⚠️ Error processing sub-territory H3 cells for {stock_point_id}: {e}")
                    continue
            
            # Calculate territory coverage
            # Reproject the main territory polygon for coverage calculation
            # Use the projected_territory_polygon calculated earlier
            if projected_territory_polygon.area > 0 and cell_geometries:
                # Union of clipped cell geometries (these are already in WGS84)
                # Reproject cell_geometries to projected CRS for union and intersection
                projected_clipped_geometries = [
                    gpd.GeoSeries([geom], crs="EPSG:4326").to_crs(PROJECTED_CRS).iloc[0]
                    for geom in cell_geometries.values()
                ]
                
                if projected_clipped_geometries:
                    cell_union_projected = unary_union(projected_clipped_geometries)
                    coverage_area_projected = projected_territory_polygon.intersection(cell_union_projected).area
                    territory_coverage = coverage_area_projected / projected_territory_polygon.area
                else:
                    territory_coverage = 0.0 # No clipped cells means no coverage
            else:
                territory_coverage = 0.0
            
            self.h3_grids[stock_point_id] = {
                'h3_resolution': resolution,
                'h3_cells': all_h3_cells,
                'clipped_cells': clipped_cells,
                'cell_geometries': cell_geometries, # These are now in original WGS84 CRS
                'territory_coverage': territory_coverage
            }
            
            print(f"✅ Generated {len(clipped_cells)} clipped cells for {stock_point_id} at resolution {resolution}")
        
        print(f"🏁 Phase 2 complete: H3 grids generated for {len(self.h3_grids)} territories")
        return self.h3_grids

    def _assign_h3_index_to_point_df(self, df: pd.DataFrame, lat_col: str, lon_col: str, id_col: str) -> pd.DataFrame:
        """
        Assigns an H3 index to each row in a DataFrame based on latitude and longitude.
        This is a utility method for H3-based analysis, not part of core Phase 1 territory definition.

        Args:
            df (pd.DataFrame): The DataFrame to process.
            lat_col (str): The name of the latitude column.
            lon_col (str): The name of the longitude column.
            id_col (str): The name of the ID column (e.g., 'customer_id', 'stock_point_id').

        Returns:
            pd.DataFrame: A DataFrame with 'h3_index' and the original ID column.
        """
        print(f"Assigning H3 index at resolution {self.h3_resolution} to {id_col} point data...")
        df_copy = df.copy()
        # Using h3.latlng_to_cell for H3 v4 compatibility
        df_copy['h3_index'] = df_copy.apply(
            lambda row: h3.latlng_to_cell(row[lat_col], row[lon_col], self.h3_resolution),
            axis=1
        )
        return df_copy[[id_col, 'h3_index', lat_col, lon_col]]

    def _get_lga_h3_mapping(self, lga_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
        """
        Generates a DataFrame mapping LGA IDs to the H3 indices that cover their polygons.
        This is a utility method for H3-based analysis, not part of core Phase 1 territory definition.

        Args:
            lga_gdf (gpd.GeoDataFrame): The GeoDataFrame containing LGA polygons.

        Returns:
            pd.DataFrame: A DataFrame with 'lga_id', 'lga_name', and 'h3_index' columns.
        """
        print(f"Generating H3 cells for LGA polygons at resolution {self.h3_resolution}...")
        lga_h3_mappings = []
        for index, row in lga_gdf.iterrows():
            lga_id = row['lga_id']
            lga_name = row['lga_name']
            polygon = row['geometry']

            if polygon.is_valid:
                # Convert shapely polygon to H3 polyfill format: [(lat, lon), ...] for exterior, then holes
                # h3.polyfill expects (lat, lon) for the tuple representation of a polygon.
                # Shapely's .coords typically return (lon, lat), so we reverse them.
                if polygon.geom_type == 'Polygon':
                    exterior = [(coord[1], coord[0]) for coord in polygon.exterior.coords]
                    interiors = [[(coord[1], coord[0]) for coord in interior.coords] for interior in polygon.interiors]
                    h3_polygon = (exterior, interiors)
                    h3_indices = h3.polyfill(h3_polygon, self.h3_resolution)
                    for h3_idx in h3_indices:
                        lga_h3_mappings.append({'lga_id': lga_id, 'lga_name': lga_name, 'h3_index': h3_idx})
                elif polygon.geom_type == 'MultiPolygon':
                    for single_polygon in polygon.geoms:
                        if single_polygon.is_valid:
                            exterior = [(coord[1], coord[0]) for coord in single_polygon.exterior.coords]
                            interiors = [[(coord[1], coord[0]) for coord in interior.coords] for interior in single_polygon.interiors]
                            h3_polygon = (exterior, interiors)
                            try:
                                h3_indices = h3.polyfill(h3_polygon, self.h3_resolution)
                                for h3_idx in h3_indices:
                                    lga_h3_mappings.append({'lga_id': lga_id, 'lga_name': lga_name, 'h3_index': h3_idx})
                            except Exception as e:
                                print(f"Error polyfilling sub-polygon of LGA {lga_name} ({lga_id}): {e}")
                        else:
                            print(f"Skipping invalid sub-polygon geometry for LGA {lga_name} ({lga_id}).")
                else:
                    print(f"Skipping LGA {lga_name} ({lga_id}) due to unsupported geometry type: {polygon.geom_type}")
            else:
                print(f"Skipping invalid LGA geometry for {lga_name} ({lga_id}).")

        return pd.DataFrame(lga_h3_mappings)

    def generate_h3_data_for_analysis(self) -> gpd.GeoDataFrame:
        """
        Generates H3-indexed data for customers, stock points, and LGAs,
        and merges them. This is a utility method for H3-based analysis,
        not the core Phase 1 territory definition.

        Returns:
            gpd.GeoDataFrame: A GeoDataFrame containing merged customer data
                              with H3-indexed stock point and LGA details.
                              Returns None if any required dataframe is missing
                              or processing fails.
        """
        print(f"\n--- Starting H3 data generation for analysis at resolution {self.h3_resolution} ---")

        if (self.customer_data is None or self.stock_point_dim is None or
            self.stock_point_lga_map is None or self.lga_gdf is None):
            print("Error: One or more required dataframes are missing for H3 data generation.")
            return None

        try:
            lga_h3_mapping = self._get_lga_h3_mapping(self.lga_gdf)
            print(f"LGA H3 mapping generated. Shape: {lga_h3_mapping.shape}")

            customer_h3_mapping = self._assign_h3_index_to_point_df(
                self.customer_data, 'latitude', 'longitude', 'customer_id'
            )
            customer_h3_mapping = pd.merge(
                customer_h3_mapping,
                self.customer_data[['customer_id', 'stock_point_id', 'geometry']],
                on='customer_id',
                how='left'
            )
            print(f"Customer H3 mapping generated. Shape: {customer_h3_mapping.shape}")

            stock_point_h3_mapping = self._assign_h3_index_to_point_df(
                self.stock_point_dim, 'latitude', 'longitude', 'stock_point_id'
            )
            stock_point_h3_mapping = pd.merge(
                stock_point_h3_mapping,
                self.stock_point_dim[['stock_point_id', 'stock_point_name']],
                on='stock_point_id',
                how='left'
            )
            print(f"Stock Point H3 mapping generated. Shape: {stock_point_h3_mapping.shape}")

            lga_customer_h3_mapping = pd.merge(
                lga_h3_mapping,
                customer_h3_mapping,
                on='h3_index',
                how='inner'
            )
            print(f"LGA-Customer H3 mapping generated. Shape: {lga_customer_h3_mapping.shape}")

            lga_stock_point_h3_mapping = pd.merge(
                lga_h3_mapping,
                stock_point_h3_mapping,
                on='h3_index',
                how='inner'
            )
            print(f"LGA-Stock Point H3 mapping generated. Shape: {lga_stock_point_h3_mapping.shape}")

            final_merged_h3_data = pd.merge(
                lga_customer_h3_mapping,
                lga_stock_point_h3_mapping.drop(columns=['lga_name']),
                on=['lga_id', 'h3_index'],
                how='outer',
                suffixes=('_customer', '_stock_point')
            )

            if 'geometry' in final_merged_h3_data.columns and isinstance(final_merged_h3_data, pd.DataFrame):
                return gpd.GeoDataFrame(
                    final_merged_h3_data,
                    geometry='geometry',
                    crs=self.customer_data.crs if hasattr(self.customer_data, 'crs') else "EPSG:4326"
                )
            else:
                print("Warning: 'geometry' column not found or not a GeoDataFrame after final merge. Cannot create GeoDataFrame.")
                return final_merged_h3_data

        except KeyError as e:
            print(f"Error during H3 data generation: Missing expected column - {e}. Please check dataframe schemas.")
            return None
        except Exception as e:
            print(f"An unexpected error occurred during H3 data generation: {e}")
            return None

    def merge_customer_data_with_stock_point_and_lga(self) -> gpd.GeoDataFrame:
        """
        (Deprecated) This method is no longer used as the primary Phase 1.
        Use define_territories for Phase 1 or generate_h3_data_for_analysis for H3-based data.
        """
        print("\n--- (Deprecated) Basic merge of customer, stock point, and LGA data ---")
        return None


# --- Example Usage with Dummy Dataframes ---
if __name__ == "__main__":
    # Create dummy dataframes mimicking the structure of your provided data
    # lga_gdf
    lga_data = {
        'state_name': ['Lagos', 'Ogun', 'Lagos', 'Oyo'],
        'state_id': [1, 2, 1, 3],
        'lga_name': ['Ikeja', 'Abeokuta South', 'Badagry', 'Ibadan North'],
        'lga_id': ['LGA001', 'LGA002', 'LGA003', 'LGA004'],
        'geometry': [
            Point(3.36, 6.59).buffer(0.01),
            Point(3.35, 7.15).buffer(0.01),
            Point(2.89, 6.42).buffer(0.01),
            Point(3.90, 7.40).buffer(0.01)
        ],
        'area_km2': [70.5, 120.3, 85.1, 150.0]
    }
    lga_gdf = gpd.GeoDataFrame(lga_data, geometry='geometry', crs="EPSG:4326")

    # sp_dim_df
    sp_dim_data = {
        'stock_point_id': ['SP001', 'SP002', 'SP003', 'SP004', 'SP005'],
        'stock_point_name': ['Ikeja Hub', 'Abeokuta Depot', 'Badagry Store', 'Lagos Central', 'Ibadan Office'],
        'latitude': [6.60, 7.16, 6.43, 6.45, 7.41],
        'longitude': [3.37, 3.36, 2.90, 3.39, 3.91]
    }
    sp_dim_df = pd.DataFrame(sp_dim_data)

    # stock_point_lga_map - SP004 and SP005 assigned to multiple LGAs to test unary_union
    sp_lga_map_data = {
        'stock_point_id': ['SP001', 'SP002', 'SP003', 'SP004', 'SP004', 'SP005'],
        'lga_id': ['LGA001', 'LGA002', 'LGA003', 'LGA001', 'LGA003', 'LGA004']
    }
    stock_point_lga_map = pd.DataFrame(sp_lga_map_data)

    # customers_gdf
    customer_data = {
        'customer_id': ['CUST001', 'CUST002', 'CUST003', 'CUST004', 'CUST005', 'CUST006'],
        'stock_point_id': ['SP001', 'SP001', 'SP002', 'SP003', 'SP004', 'SP005'],
        'longitude': [3.38, 3.37, 3.35, 2.91, 3.40, 3.92],
        'latitude': [6.61, 6.60, 7.17, 6.44, 6.46, 7.42],
        'geometry': [
            Point(3.38, 6.61), Point(3.37, 6.60), Point(3.35, 7.17),
            Point(2.91, 6.44), Point(3.40, 6.46), Point(3.92, 7.42)
        ]
    }
    customers_gdf = gpd.GeoDataFrame(customer_data, geometry='geometry', crs="EPSG:4326")

    print("--- Dummy DataFrames Created ---")
    print("\nlga_gdf sample:\n", lga_gdf.head())
    print("\nsp_dim_df sample:\n", sp_dim_df.head())
    print("\nstock_point_lga_map sample:\n", stock_point_lga_map.head())
    print("\ncustomers_gdf sample:\n", customers_gdf.head())

    # Instantiate the clusterer
    clusterer = H3SpatialClusterer(
        customer_data=customers_gdf,
        stock_point_dim=sp_dim_df,
        stock_point_lga_map=stock_point_lga_map,
        lga_gdf=lga_gdf,
        h3_resolution=7
    )

    # --- Run Phase 1: Territory Definition ---
    defined_territories = clusterer.define_territories()

    if defined_territories:
        print("\n--- Phase 1: Defined Territories Summary ---")
        for sp_id, territory_info in defined_territories.items():
            print(f"Stock Point ID: {sp_id}")
            print(f"  LGA Count: {territory_info['lga_count']}")
            print(f"  Is Contiguous: {territory_info['is_contiguous']}")
            print(f"  Sub-territories: {len(territory_info['sub_territories'])}")
            print(f"  Total Area (km²): {territory_info['total_area_km2']:.2f}")
            print(f"  Validation Status: {territory_info['validation_status']}")
        print(f"\nTotal territories defined: {len(defined_territories)}")
    else:
        print("\nPhase 1 territory definition failed.")

    # --- Run Phase 2: H3 Grid Generation ---
    if defined_territories: # Only run Phase 2 if Phase 1 was successful
        h3_grid_results = clusterer.generate_h3_grids(defined_territories)

        if h3_grid_results:
            print("\n--- Phase 2: H3 Grid Generation Results Summary ---")
            for sp_id, grid_info in h3_grid_results.items():
                print(f"Stock Point ID: {sp_id}")
                print(f"  H3 Resolution: {grid_info['h3_resolution']}")
                print(f"  Total H3 Cells (raw): {len(grid_info['h3_cells'])}")
                print(f"  Clipped H3 Cells: {len(grid_info['clipped_cells'])}")
                print(f"  Territory Coverage: {grid_info['territory_coverage']:.2%}")
            print(f"\nTotal territories with H3 grids: {len(h3_grid_results)}")
        else:
            print("\nPhase 2 H3 grid generation failed.")
    else:
        print("\nSkipping Phase 2 as Phase 1 did not define territories successfully.")
